<a href="https://colab.research.google.com/github/shiosabax/test_web_program/blob/shiosabax-patch-1/kabuka_sihyou.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import requests
import json

# 1. 取得したい主要株価指数のシンボル（識別子）を「リスト」で定義
# ^N225: 日経平均株価, ^TOPX: TOPIX, ^GSPC: S&P500, ^IXIC: NASDAQ総合
indices = ["^N225", "^TOPX", "^GSPC", "^IXIC"]

# サーバーにブラウザからのアクセスであることを伝えるヘッダー情報
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

print("=== 主要株価指数のデータ取得開始 ===")

# 2. リストの中身を for ループで1つずつ順番に処理
for symbol in indices:
    # データを取得するURL（Yahoo Financeの公開エンドポイント形式）
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}?interval=1d&range=1d"

    # requestsを使ってデータを取得
    response = requests.get(url, headers=headers)

    # 通信が成功（ステータスコード 200）したか確認
    if response.status_code == 200:
        data = response.json()

        try:
            meta = data["chart"]["result"][0]["meta"]
            name = meta.get("shortName", symbol)
            currency = meta.get("currency", "")
            current_price = meta.get("regularMarketPrice", 0)
            prev_close = meta.get("chartPreviousClose", current_price)

            # 前日比と変動率の計算
            diff = current_price - prev_close
            diff_percent = (diff / prev_close) * 100 if prev_close else 0

            # 結果を分かりやすく出力
            sign = "+" if diff >= 0 else ""
            print(f"【{name} ({symbol})】")
            print(f"  最新値: {current_price:,.2f} {currency}")
            print(f"  前日比: {sign}{diff:,.2f} ({sign}{diff_percent:.2f}%)\n")

        except (KeyError, IndexError) as e:
            print(f"【{symbol}】データの解析に失敗しました。")
    else:
        print(f"【{symbol}】データの取得に失敗しました（ステータス: {response.status_code}）")

    # サーバー負荷軽減のため1秒待機
    time.sleep(1)

print("=== 取得完了 ===")

=== 主要株価指数のデータ取得開始 ===
【Nikkei 225 (^N225)】
  最新値: 66,405.56 JPY
  前日比: +273.58 (+0.41%)

【^TOPX (^TOPX)】
  最新値: 0.00 None
  前日比: +0.00 (+0.00%)

【S&P 500 (^GSPC)】
  最新値: 7,711.76 USD
  前日比: -19.23 (-0.25%)

【NASDAQ Composite (^IXIC)】
  最新値: 26,402.42 USD
  前日比: -138.93 (-0.52%)

=== 取得完了 ===


In [4]:
import time
import requests
import pandas as pd
from google.colab import files

indices = ["^N225", "^TOPX", "^GSPC", "^IXIC"]
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 取得したデータをためておく空のリストを用意
data_list = []

print("=== 主要株価指数のデータ取得中... ===")

for symbol in indices:
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}?interval=1d&range=1d"
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        try:
            meta = data["chart"]["result"][0]["meta"]
            name = meta.get("shortName", symbol)
            currency = meta.get("currency", "")
            current_price = meta.get("regularMarketPrice", 0)
            prev_close = meta.get("chartPreviousClose", current_price)

            diff = current_price - prev_close
            diff_percent = (diff / prev_close) * 100 if prev_close else 0

            # 1件分のデータを辞書形式にしてリストに追加 (append)
            data_list.append({
                "シンボル": symbol,
                "指数名": name,
                "最新値": round(current_price, 2),
                "通貨": currency,
                "前日比": round(diff, 2),
                "変動率(%)": round(diff_percent, 2)
            })
        except (KeyError, IndexError):
            pass

    time.sleep(1)

# リストからデータフレーム（表形式）を作成
df = pd.DataFrame(data_list)
print("\n--- 取得結果の一覧 ---")
print(df)

# CSVファイルとして書き出し
csv_filename = "stock_indices.csv"
df.to_csv(csv_filename, index=False, encoding="utf_8_sig")
print(f"\n『{csv_filename}』として保存しました。")

# Google Colabからパソコンへ直接ダウンロード（必要に応じて実行）
# files.download(csv_filename)

=== 主要株価指数のデータ取得中... ===

--- 取得結果の一覧 ---
    シンボル               指数名       最新値    通貨     前日比  変動率(%)
0  ^N225        Nikkei 225  66405.56   JPY  273.58    0.41
1  ^TOPX             ^TOPX      0.00  None    0.00    0.00
2  ^GSPC           S&P 500   7711.76   USD  -19.23   -0.25
3  ^IXIC  NASDAQ Composite  26402.42   USD -138.93   -0.52

『stock_indices.csv』として保存しました。


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>